CLEANING OF RAW DATASET

1. Loading the Dataset

In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("ckd-dataset-v2.csv")

2. Inspection of Dataset

In [3]:
print("Initial Shape:", df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum())

Initial Shape: (202, 29)
  bp (Diastolic)  bp limit             sg        al     class       rbc  \
0       discrete  discrete       discrete  discrete  discrete  discrete   
1            NaN       NaN            NaN       NaN       NaN       NaN   
2              0         0  1.019 - 1.021    01-Jan       ckd         0   
3              0         0  1.009 - 1.011       < 0       ckd         0   
4              0         0  1.009 - 1.011     â‰¥ 4       ckd         1   

         su        pc       pcc        ba  ...       htn        dm       cad  \
0  discrete  discrete  discrete  discrete  ...  discrete  discrete  discrete   
1       NaN       NaN       NaN       NaN  ...       NaN       NaN       NaN   
2       < 0         0         0         0  ...         0         0         0   
3       < 0         0         0         0  ...         0         0         0   
4       < 0         1         0         1  ...         0         0         0   

      appet        pe       ane            

3. Removing Metadata Rows

In [4]:
df = df[df["bp (Diastolic)"] != "discrete"]
df = df[df["age"] != "meta"]
df.dropna(how="all", inplace=True)

4. Resetting the index

In [5]:
df.reset_index(drop=True, inplace=True)
print("After Removing Metadata Rows:", df.shape)
print(df.head())

After Removing Metadata Rows: (200, 29)
  bp (Diastolic) bp limit             sg      al class rbc   su pc pcc ba  \
0              0        0  1.019 - 1.021  01-Jan   ckd   0  < 0  0   0  0   
1              0        0  1.009 - 1.011     < 0   ckd   0  < 0  0   0  0   
2              0        0  1.009 - 1.011   â‰¥ 4   ckd   1  < 0  1   0  1   
3              1        1  1.009 - 1.011  03-Mar   ckd   0  < 0  0   0  0   
4              0        0  1.015 - 1.017     < 0   ckd   0  < 0  0   0  0   

   ... htn dm cad appet pe ane                grf stage affected     age  
0  ...   0  0   0     0  0   0        â‰¥ 227.944    s1        1    < 12  
1  ...   0  0   0     0  0   0        â‰¥ 227.944    s1        1    < 12  
2  ...   0  0   0     1  0   0  127.281 - 152.446    s1        1    < 12  
3  ...   0  0   0     0  0   0  127.281 - 152.446    s1        1    < 12  
4  ...   0  1   0     1  1   0  127.281 - 152.446    s1        1  Dec-20  

[5 rows x 29 columns]


5. Fixing certain encoding issues

In [19]:
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].str.replace("â‰¥", ">=", regex=False)
        df[col] = df[col].str.replace("â‰¤", "<=", regex=False)
        df[col] = df[col].str.replace("≥", ">=", regex=False)
        df[col] = df[col].str.replace("≤", "<=", regex=False)

6. Replace special missing values

In [20]:
df.replace("?", np.nan, inplace=True)
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].str.strip()

7. Handling uneven range values

In [22]:
def convert_range_to_mean(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = value.replace("â‰¥", ">=").replace("â‰¤", "<=")

    special_map = {"01-Jan": 1,"02-Feb": 2,"03-Mar": 3,"04-Apr": 4,"05-May": 5,"06-Jun": 6,"07-Jul": 7,"08-Aug": 8,"09-Sep": 9,"10-Oct": 10,"11-Nov": 11,"12-Dec": 12,
        "Dec-20": 20}

    if value in special_map:
        return special_map[value]

    if value.startswith(">="):
        try:
            return float(value.replace(">=", "").strip())
        except:
            return np.nan

    if value.startswith("<"):
        try:
            return float(value.replace("<", "").strip())
        except:
            return np.nan

    if " - " in value:
        try:
            low, high = value.split(" - ")
            return (float(low.strip()) + float(high.strip())) / 2
        except:
            return np.nan

    try:
        return float(value)
    except:
        return np.nan


range_columns = [
    "sg", "al", "su", "grf", "age",
    "bgr", "bu", "sod", "sc", "pot",
    "hemo", "pcv", "rbcc", "wbcc"
]

for col in range_columns:
    if col in df.columns:
        df[col] = df[col].apply(convert_range_to_mean)

8. Conversion of Numeric Columns

In [23]:
numeric_columns = ["bp (Diastolic)","bp limit","sg","al","rbc","su","pc","pcc","ba","bgr","bu","sod","sc","pot","hemo","pcv","rbcc","wbcc","htn","dm","cad","appet",
    "pe","ane","grf","affected","age"]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 29 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   bp (Diastolic)  200 non-null    int64  
 1   bp limit        200 non-null    int64  
 2   sg              200 non-null    float64
 3   al              200 non-null    float64
 4   class           200 non-null    object 
 5   rbc             200 non-null    int64  
 6   su              186 non-null    float64
 7   pc              200 non-null    int64  
 8   pcc             200 non-null    int64  
 9   ba              200 non-null    int64  
 10  bgr             0 non-null      float64
 11  bu              0 non-null      float64
 12  sod             0 non-null      float64
 13  sc              0 non-null      float64
 14  pot             0 non-null      float64
 15  hemo            0 non-null      float64
 16  pcv             0 non-null      float64
 17  rbcc            0 non-null      flo

9. Handling missing values

In [24]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])